In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install fair-esm

In [ ]:
!pip install biopython

In [ ]:
import torch
import esm
import os
import numpy as np
import pandas as pd
from Bio import PDB

# Base path for the folds
base_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
new_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method'

In [ ]:
# Load the pre-trained ESM-2 model and its alphabet
esm2_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
esm2_model.eval()  # Disables dropout for deterministic results

In [ ]:
def get_esm2_embedding(sequence, model, alphabet):
    batch_converter = alphabet.get_batch_converter()
    data = [("protein", sequence)]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[6], return_contacts=True)
    token_representations = results["representations"][6]

    sequence_embedding = token_representations[0, 1 : batch_lens[0] - 1]
    return sequence_embedding

In [ ]:
# Map three-letter amino acid codes to one-letter codes
aa_map = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

# Process each fold's training set
for fold in range(1, 6):
    input_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_info_SI_interaction')
    output_dir = os.path.join(new_fold_path, f'fold_{fold}', 'data_test', 'all_antigen_info_SI')
    os.makedirs(output_dir, exist_ok=True)

    # Process each TSV file in the input directory
    for tsv_filename in os.listdir(input_dir):
        if tsv_filename.endswith('.tsv'):
            file_path = os.path.join(input_dir, tsv_filename)
            df = pd.read_csv(file_path, sep='\t')

            # Convert three-letter AA codes to one-letter sequence
            sequence = ''.join(df['AA'].map(aa_map))

            # Get the embeddings for the sequence
            embeddings = get_esm2_embedding(sequence, esm2_model, alphabet)

            embedding_df = pd.DataFrame(embeddings, columns=[f'embedding_{i}' for i in range(embeddings.shape[1])])
            df = pd.concat([df, embedding_df], axis=1)

            output_file = os.path.join(output_dir, f"{tsv_filename}_with_embeddings.tsv")
            df.to_csv(output_file, sep='\t', index=False)


In [ ]:
import os
import pandas as pd
import numpy as np

# Process each fold's output data
for fold in range(1, 6):
    output_dir = os.path.join(new_fold_path, f'fold_{fold}', 'data_test', 'all_antigen_info_SI')
    os.makedirs(output_dir, exist_ok=True)

    # List to hold all dataframes
    dfs = []

    # Iterate through each TSV file in the output directory and read them into a dataframe
    for tsv_filename in os.listdir(output_dir):
        tsv_filepath = os.path.join(output_dir, tsv_filename)
        if tsv_filename.endswith('.tsv'):
            df = pd.read_csv(tsv_filepath, sep='\t')
            dfs.append(df)

    # Concatenate all dataframes into a single dataframe
    all_data_df = pd.concat(dfs, ignore_index=True)

    # Interacting info of antigen columns
    ab_cols = ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N',
               'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']

    # Add Status column
    all_data_df['Status'] = np.where(
        all_data_df[ab_cols].sum(axis=1) == 0,
        'Non-Interacting',
        'Interacting'
    )

    # Filter out non-standard amino acid codes
    standard_amino_acids = [
        'ALA', 'CYS', 'ASP', 'GLU', 'PHE',
        'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
        'MET', 'ASN', 'PRO', 'GLN', 'ARG',
        'SER', 'THR', 'VAL', 'TRP', 'TYR'
    ]

    # Filter the dataframe
    all_data_df = all_data_df[all_data_df['AA'].isin(standard_amino_acids)]

    # Define the directory where the merged file will be saved
    save_dir = os.path.join(new_fold_path, f'fold_{fold}', 'data_test')
    os.makedirs(save_dir, exist_ok=True)

    # Save the concatenated dataframe to a CSV file
    all_data_with_si_filepath = os.path.join(save_dir, 'all_data_with_SI.csv')
    all_data_df.to_csv(all_data_with_si_filepath, index=False)
    print(f"All data has been merged and saved to {all_data_with_si_filepath}")
